# Teaching ORCA trajectory models

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sthsci/Orca/blob/main/notebooks/03_trajectory_model_tutorial.ipynb)

**Teaching notebook.** Simulate ordered successful and unsuccessful contacts, summarise decision states, and optionally compare stable heterogeneity with history dependence.

Run the cells from top to bottom. Values collected near the start of each notebook are safe places to experiment. Bayesian SMC fitting is deliberately disabled by default in the analysis notebooks because it can take several minutes; set `RUN_INFERENCE = True` when the data checks and descriptive plots look right.

Use synthetic or approved anonymised data only. Do not upload names, clinical metadata, raw microscopy, or a donor key that could identify participants.


## 1. Why order matters

Total kills discard the order of contacts. A history such as `0,0,1,0` records two unsuccessful contacts, one successful contact, then another unsuccessful contact. ORCA trajectory models ask two different questions:

- **Stable heterogeneity:** do cells have persistently different baseline killing propensities ($\sigma_\eta>0$)?
- **History dependence:** does the probability of the next success change after previous failures ($\beta_f$) or successes ($\beta_s$)?

Combining homogeneous/heterogeneous with history-independent/history-dependent assumptions produces four candidate models.


In [ ]:
from pathlib import Path
import importlib.util
import subprocess
import sys


def find_orca_checkout():
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "src" / "bayesorca").is_dir():
            return candidate
    return None


ORCA_ROOT = find_orca_checkout()
if ORCA_ROOT is not None:
    sys.path[:0] = [str(ORCA_ROOT), str(ORCA_ROOT / "src")]
elif importlib.util.find_spec("bayesorca") is None:
    if sys.version_info[:2] != (3, 12):
        raise RuntimeError("ORCA currently requires a Python 3.12 Colab runtime.")
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "git+https://github.com/sthsci/Orca.git@main",
        ]
    )

import bayesorca

print("bayesorca", bayesorca.__version__)
print("Python", sys.version.split()[0])


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from bayesorca.trajectories import (
    TRAJECTORY_MODEL_SPECS,
    TrajectorySettings,
    expanded_trajectory_frame,
    run_trajectory_conditions,
    simulate_trajectory_frame,
    trajectory_evidence_frame,
    trajectory_summary_frame,
)

trajectories, truth = simulate_trajectory_frame(
    condition="Synthetic",
    n_cells=120,
    mu_lambda=4.0,
    sigma_lambda=2.0,
    p0=0.20,
    sigma_eta=0.75,
    beta_f=0.8,
    beta_s=-0.8,
    observation_time=1.0,
    seed=2026,
)
print("Generating model:", truth["Synthetic"]["true_model_key"])
trajectories.head()


In [ ]:
contacts = expanded_trajectory_frame(trajectories)
state_summary = (
    contacts.groupby(
        ["previous_nonlethal_contacts", "previous_lethal_contacts"],
        as_index=False,
    )
    .agg(contacts=("outcome", "size"), kill_probability=("outcome", "mean"))
)

fig, ax = plt.subplots(figsize=(8, 5.5))
points = ax.scatter(
    state_summary["previous_nonlethal_contacts"],
    state_summary["previous_lethal_contacts"],
    s=25 + 5 * state_summary["contacts"],
    c=state_summary["kill_probability"],
    cmap="viridis",
    vmin=0,
    vmax=1,
    alpha=0.85,
)
fig.colorbar(points, ax=ax, label="Observed next-contact success fraction")
ax.set(
    xlabel="Previous unsuccessful contacts",
    ylabel="Previous successful contacts",
    title="Empirical decision states (point size = contacts observed)",
)
plt.show()

state_summary.sort_values("contacts", ascending=False).head(12)


## 2. Optional model comparison

Empirical state fractions mix true history effects, cell-to-cell differences, and uneven sampling. The trajectory likelihood models them jointly. The preview below is intentionally small; increase particles, chains, and quadrature points for a scientific analysis.


In [ ]:
RUN_INFERENCE = False

if RUN_INFERENCE:
    settings = TrajectorySettings(
        draws=128,
        chains=1,
        cores=1,
        seed=2026,
        n_quad=10,
    )
    results = run_trajectory_conditions(
        trajectories,
        observation_time=1.0,
        settings=settings,
        model_keys=list(TRAJECTORY_MODEL_SPECS),
    )
    display(trajectory_evidence_frame(results))
    display(trajectory_summary_frame(results))
else:
    print("Set RUN_INFERENCE = True to fit the four trajectory models.")


## Interpretation checklist

- A blank history is a valid cell with zero observed contacts; keep it in the input.
- `beta_f` and `beta_s` describe associations with prior outcomes after accounting for the modelled baseline structure.
- The order of the binary history must match experimental time.
- Sparse states are noisy; point size above shows how many decisions support each empirical fraction.
- Compare all four models before describing an effect as stable heterogeneity or history dependence.
